In [2]:
import pandas as pd

In [3]:
decomposition_df = pd.read_json(path_or_buf="dictionary.txt", lines=True)
decomposition_df = decomposition_df[["character", "decomposition", "etymology"]]

In [5]:
hanzii_df = pd.read_csv(filepath_or_buffer="hanziDB.csv")
hanzii_df.drop(["general_standard_num"], axis=1, inplace=True)

In [6]:
joined_df = pd.merge(decomposition_df, hanzii_df, how="right", on="character")
joined_df = joined_df[['character', 'radical', 'decomposition', 'pinyin', 'definition', 'etymology', 'stroke_count', 'radical_code', 'frequency_rank', 'hsk_levl']]

In [7]:
mask = joined_df["radical"].notna() & joined_df["radical"].map(lambda x: isinstance(x, str))
joined_df = joined_df.loc[mask].reset_index(drop=True)
joined_df = joined_df.drop_duplicates(subset=['character', 'radical'])
joined_df.to_csv(path_or_buf="hanzii_db.csv", index=False)

In [8]:
radicals_df = pd.read_csv(filepath_or_buffer="hanzii_db.csv")
s = radicals_df["radical"].dropna()
s = s[s.map(lambda x: isinstance(x, str))]

radicals = s.unique()

In [18]:
import numpy as np
import json

df = pd.read_csv("hanzii_db.csv")

radicals_to_vec = {r: np.eye(len(radicals))[i].tolist() for i, r in enumerate(radicals)}
with open("radical_list.json", "w", encoding="utf-8") as f:
    f.write("{\n")
    for i, (k, v) in enumerate(radicals_to_vec.items()):
        f.write(f'  "{k}": {json.dumps(v, ensure_ascii=False)}')
        if i < len(radicals_to_vec) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("}\n")

vectors = []
missing_radicals = set()

for radical in df["radical"]:
    if radical in radicals_to_vec:
        vectors.append(radicals_to_vec[radical])
    else:
        vectors.append(np.zeros(len(radicals_to_vec)))  # fallback
        missing_radicals.add(radical)

numpy_vectors = np.array(vectors)
np.save("radical_vectors.npy", numpy_vectors)

In [25]:
import os, json, numpy as np, pandas as pd

# __file__ is not defined in interactive environments (like Jupyter).
# Fall back to the current working directory when needed.
try:
    BASE_DIR = os.path.dirname(os.path.dirname(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HANZII_FILE = os.path.join(BASE_DIR, "hanzii_db.csv")
RADICAL_LIST = os.path.join(BASE_DIR, "radical_list.json")
WORD_LIST = os.path.join(BASE_DIR, "word_list.json")

hanzii_df = pd.read_csv(HANZII_FILE)

with open("learned_words.txt", "r", encoding="utf-8") as file:
    word_list = [w.strip() for w in file if w.strip()]
    
with open(RADICAL_LIST, encoding="utf-8") as f:
    radical_list = json.load(f)

In [26]:
def get_word_vector(word, radical_list, agg="mean"):
    vs = []
    for char in word:
        
        # Tìm bộ thủ trong chữ
        row = hanzii_df.loc[hanzii_df["character"] == char]
        if row.empty:
            raise ValueError(f"Không tìm thấy chữ {char}")
        radical = row["radical"].values[0]
        
        # Lấy vector bộ thủ
        if radical in radical_list:
            vs.append(radical_list[radical])
            
    if not vs:
        return None
    
    vs = np.stack(vs)
    return vs.mean(axis=0) if agg == "mean" else vs.sum(axis=0)

In [27]:
vectors = []
valid_words = []

for w in word_list:
    v = get_word_vector(w, radical_list)
    if v is not None:
        valid_words.append(w)
        vectors.append(v)

word_list = valid_words
word_vectors = np.stack(vectors)

with open("word_list.json", "w", encoding="utf-8") as f:
    json.dump(word_list, f, ensure_ascii=False, indent=2)

np.save("word_vectors.npy", word_vectors)
